In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# تحميل ملفات التدريب والاختبار (تأكد من وجود train.csv و test.csv لديك)
train_df = pd.read_csv('Spaceship_TitanicTrain.csv')
test_df = pd.read_csv('test.csv')

print("--- Train Data Shape ---")
print(train_df.shape)

print("\n--- Test Data Shape ---")
print(test_df.shape)

print("\n--- Missing Values in Train Data ---")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

# استعراض أول 5 صفوف من بيانات التدريب
display(train_df.head())

--- Train Data Shape ---
(8693, 14)

--- Test Data Shape ---
(4277, 13)

--- Missing Values in Train Data ---
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
dtype: int64


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
# 1. دمج الـ Train والـ Test  خطوات المعالجة والهندسة
df_all = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# 2. هندسة ميزات الكابينة (Cabin -> Deck, Num, Side)
df_all[['Deck', 'Cabin_Num', 'Side']] = df_all['Cabin'].str.split('/', expand=True)
df_all.drop('Cabin', axis=1, inplace=True)

# 3. استخراج رقم المجموعة وعائلة الراكب من PassengerId
df_all['Group'] = df_all['PassengerId'].apply(lambda x: x.split('_')[0])
df_all['Group_Size'] = df_all['Group'].map(df_all['Group'].value_counts())

# 4. تجميع إجمالي الفواتير والخدمات في عمود واحد (Total Spending)
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    df_all[col] = df_all[col].fillna(0)

df_all['Total_Spending'] = df_all[spend_cols].sum(axis=1)
df_all['No_Spending'] = (df_all['Total_Spending'] == 0).astype(int)

print("Feature Engineering completed successfully!")
print("New Data Shape:", df_all.shape)

Feature Engineering completed successfully!
New Data Shape: (12970, 20)


In [4]:
# 1. تعبئة القيم المفقودة للأعمدة العددية والعامة
df_all['Age'] = df_all['Age'].fillna(df_all['Age'].median())
df_all['HomePlanet'] = df_all['HomePlanet'].fillna('Unknown')
df_all['Destination'] = df_all['Destination'].fillna('Unknown')
df_all['Deck'] = df_all['Deck'].fillna('Unknown')
df_all['Side'] = df_all['Side'].fillna('Unknown')
df_all['Cabin_Num'] = df_all['Cabin_Num'].fillna(0).astype(int)

# للأعمدة المنطقية (Boolean) نعوض الفراغات بأكثر قيمة تكراراً أو False
for col in ['CryoSleep', 'VIP']:
    df_all[col] = df_all[col].fillna(False).astype(int)

# تحويل عمود التوجيه Target (Transported) إلى أرقام إذا كان موجوداً
if 'Transported' in df_all.columns:
    df_all['Transported'] = df_all['Transported'].astype(float)

# 2. إسقاط الأعمدة غير الضرورية للتدريب مباشرة
df_all.drop(['PassengerId', 'Name', 'Group'], axis=1, inplace=True, errors='ignore')

# 3. ترميز كافة الأعمدة النصية (One-Hot Encoding)
df_all_encoded = pd.get_dummies(df_all, drop_first=True, dtype=int)

# 4. إعادة فصل البيانات إلى Train و Test بناءً على الحجم الأصلي (8693 صف للتدريب)
train_processed = df_all_encoded.iloc[:8693, :]
test_processed = df_all_encoded.iloc[8693:, :].drop('Transported', axis=1, errors='ignore')

# فصل X و y لبيانات التدريب الخاصة بنا
X = train_processed.drop('Transported', axis=1)
y = train_processed['Transported']

print("Data preprocessing and encoding completed successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)

Data preprocessing and encoding completed successfully!
X shape: (8693, 28)
y shape: (8693,)


/tmp/ipykernel_934/3441107089.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_all[col] = df_all[col].fillna(False).astype(int)
/tmp/ipykernel_934/3441107089.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_all[col] = df_all[col].fillna(False).astype(int)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. تقسيم البيانات محلياً للتدريب والاختبار بنسبة 80/20
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. تعريف وتقسيم الموديلات القوية
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='logloss')
}

print("--- Spaceship Titanic Models Evaluation ---\n")

# 3. التدريب والتقييم
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)

    print(f"Model: {name}")
    print(f"   Validation Accuracy: {acc * 100:.2f}%")
    print("-" * 40)

--- Spaceship Titanic Models Evaluation ---

Model: Random Forest
   Validation Accuracy: 80.74%
----------------------------------------
Model: XGBoost
   Validation Accuracy: 81.48%
----------------------------------------
